In [23]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np

# load the Tiny Shakespeare dataset
dataset, info = tfds.load('tiny_shakespeare', with_info=True, as_supervised=False)

In [24]:
# get the text from the dataset
text = next(iter(dataset['train']))['text'].numpy().decode('utf-8')


In [25]:
# create a mapping from unique characters to indices
vocab = sorted(set(text))
char2idx = {char: idx for idx, char in enumerate(vocab)}
idx2char = np.array(vocab)

In [26]:
# numerically represent the characters
text_as_int = np.array([char2idx[c] for c in text])

In [27]:
# create training examples and targets
seq_length = 100
examples_per_epoch = len(text) // (seq_length + 1)


In [28]:
# create training sequences
char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)

sequences = char_dataset.batch(seq_length + 1, drop_remainder=True)

In [29]:
def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text

dataset = sequences.map(split_input_target)

In [30]:
# batch size and buffer size
BATCH_SIZE = 64
BUFFER_SIZE = 10000

dataset = (
    dataset
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.experimental.AUTOTUNE)
)

In [31]:
# length of the vocabulary
vocab_size = len(vocab)

# the embedding dimension
embedding_dim = 256

# number of RNN units
rnn_units = 1024


In [32]:
def build_model(vocab_size, embedding_dim, rnn_units, batch_size):
    model = tf.keras.Sequential([
        tf.keras.layers.Embedding(vocab_size, embedding_dim, input_length=None), # Removed batch_input_shape and set input_length to None for variable sequence length
        tf.keras.layers.LSTM(rnn_units, return_sequences=True, stateful=True, recurrent_initializer='glorot_uniform'),
        tf.keras.layers.Dense(vocab_size)
    ])
    return model

model = build_model(vocab_size, embedding_dim, rnn_units, BATCH_SIZE)

In [43]:
def loss(labels, logits):
    return tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)

model.compile(optimizer='adam', loss=loss)

In [44]:
import os

# directory where the checkpoints will be saved
checkpoint_dir = './training_checkpoints'

# name of the checkpoint files
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt_{epoch}.weights.h5") # Added '.weights.h5' to the filename

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_prefix,
    save_weights_only=True
)

# train the model
EPOCHS = 10
history = model.fit(dataset, epochs=EPOCHS, callbacks=[checkpoint_callback])

Epoch 1/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 14s 67ms/step - loss: 2.8720
Epoch 2/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 21s 68ms/step - loss: 1.8670
Epoch 3/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 20s 69ms/step - loss: 1.6014
Epoch 4/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 12s 69ms/step - loss: 1.4782
Epoch 5/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 21s 71ms/step - loss: 1.3992
Epoch 6/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 21s 72ms/step - loss: 1.3481
Epoch 7/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 20s 71ms/step - loss: 1.3037
Epoch 8/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 13s 72ms/step - loss: 1.2672
Epoch 9/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 13s 74ms/step - loss: 1.2328
Epoch 10/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 20s 73ms/step - loss: 1.1969


In [46]:
# Build the model
model = build_model(vocab_size, embedding_dim, rnn_units, batch_size=1)

# Find the latest checkpoint manually
checkpoint_dir = './training_checkpoints'
checkpoint_files = [f for f in os.listdir(checkpoint_dir) if f.endswith('.weights.h5')]
if checkpoint_files:
    latest_checkpoint = sorted(checkpoint_files)[-1]  # Get the latest checkpoint file
    latest_checkpoint_path = os.path.join(checkpoint_dir, latest_checkpoint)
    print("Loading weights from:", latest_checkpoint_path)

    # Build the model with an input shape to initialize the layers before loading weights
    # This ensures the layers have the correct shapes and variables
    model.build(tf.TensorShape([1, None]))

    model.load_weights(latest_checkpoint_path)
else:
    print("No checkpoint files found in:", checkpoint_dir)

# The model is already built in the previous step
# model.build(tf.TensorShape([1, None]))

Loading weights from: ./training_checkpoints/ckpt_9.weights.h5


In [52]:
def generate_text(model, start_string):
    num_generate = 1000

    input_eval = [char2idx[s] for s in start_string]
    input_eval = tf.expand_dims(input_eval, 0)

    text_generated = []

    # Reset the states of the LSTM layer directly
    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.LSTM):
            layer.reset_states()

    for i in range(num_generate):
        predictions = model(input_eval)
        predictions = tf.squeeze(predictions, 0)

        predicted_id = tf.random.categorical(predictions, num_samples=1)[-1, 0].numpy()
        input_eval = tf.expand_dims([predicted_id], 0)

        text_generated.append(idx2char[predicted_id])

    return (start_string + ''.join(text_generated))

In [53]:
print(generate_text(model, start_string=u"QUEEN: So, lets end this"))

QUEEN: So, lets end this sudden:
So from this news is not reasons;
And I have pass'd the raiges of my exraced.

KING RICHARD III:
Read the law, to be sleep! I am gone;
And, for your voices acts drown'd with sorriegry,
I never may was't;
And what ie the coust to rive,
To whigh vanity.

ARCHIDYAM:
Nay, by the earth: your broken managoned straight
But Romeo!

Shepherd:
Wester!

ELBOW:
Be consulZourselves more wonseread than his.

EDWARD:
'Ton offices, peryent, or no more.

GRUMIO:
Too more, that's he? but that?

DUKE OF YORK:
Why, Wart like thee, sheep Joint Preserping,
To see him lease.

CLARENCE:
Ay, to be the idlonered.

LADY CAPULET:
Well, sir! I cannot.

HERMIONE:
See you weary is so much for her fellow?

ROMEO:
It is to say one if you may as well;
For they could feel, and do him go all
Shake him 's burntancly shall be at Bobe of Juliet.

LADY CAPULET:
Where's the mannerous in the old bottery
Who victor's toences.

SICINIUS:
You are too find:
Be, put me oq; for cease poor-brack, and 